<a href="https://colab.research.google.com/github/graemewales13/Langchain-Basics/blob/main/05-agents-intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aurelio-labs/langchain-course/blob/main/chapters/05-agents-intro.ipynb)

#### LangChain Essentials Course

# LangChain Agents Intro

LangChain is one of the most popular open source libraries for AI Engineers. It's goal is to abstract away the complexity in building AI software, provide easy-to-use building blocks, and make it easier when switching between AI service providers.

In this example, we will introduce LangChain's Agents, adding the ability to use tools such as search and calculators to complete tasks that normal LLMs cannot fufil. In this example we will be using OpenAI's `gpt-4o-mini`.

In [1]:
!pip install -U \
  "langchain==1.2.17" \
  "langchain-core==1.3.3" \
  "langchain-openai==1.2.1" \
  "langchain-community==0.4.1" \
  "langsmith>=0.3.45" \
  "serpapi"

  Using cached serpapi-1.0.2-py3-none-any.whl.metadata (12 kB)
Using cached serpapi-1.0.2-py3-none-any.whl (11 kB)


---

> ⚠️ We will be using OpenAI for this example allowing us to run everything via API. If you would like to use Ollama instead, check out the [Ollama LangChain Course](https://github.com/aurelio-labs/langchain-course/tree/main/notebooks/ollama).

---

---

> ⚠️ If using LangSmith, add your API key below:

In [5]:
import os
from getpass import getpass

os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY") or \
    getpass("Enter LangSmith API Key: ")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "aurelioai-langchain-course-agents-intro-openai"

Enter LangSmith API Key: ··········


---

## Introduction to Tools

Tools are a way augment our LLMs with code execution. A tool is simply a function formatted so that our agent can undertstand how to use it, and then execute it. Let's start by creating a few simple tools.

We can use the `@tool` decorator to create an LLM-compatible tool from a standard python function — this function should include a few things for optimal performance:

* A docstring describing what the tool does and when it should be used, this will be read by our LLM/agent and used to decide when to use the tool, and also how to use the tool.

* Clear parameter names that ideally tell the LLM what each parameter is, if it isn't clear we make sure the docstring explains what the parameter is for and how to use it.

* Both parameter and return type annotations.

In [6]:
from langchain_core.tools import tool

@tool
def add(x: float, y: float) -> float:
    """Add 'x' and 'y'."""
    return x + y

@tool
def multiply(x: float, y: float) -> float:
    """Multiply 'x' and 'y'."""
    return x * y

@tool
def exponentiate(x: float, y: float) -> float:
    """Raise 'x' to the power of 'y'."""
    return x ** y

@tool
def subtract(x: float, y: float) -> float:
    """Subtract 'x' from 'y'."""
    return y - x

With the `@tool` decorator our function is turned into a `StructuredTool` object, which we can see below:

In [7]:
multiply

StructuredTool(name='multiply', description="Multiply 'x' and 'y'.", args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x7c5a0bb6dc60>)

We can see the tool name, description, and arg schema:

In [8]:
print(f"{add.name=}\n{add.description=}")

add.name='add'
add.description="Add 'x' and 'y'."


In [9]:
add.args_schema.model_json_schema()

{'description': "Add 'x' and 'y'.",
 'properties': {'x': {'title': 'X', 'type': 'number'},
  'y': {'title': 'Y', 'type': 'number'}},
 'required': ['x', 'y'],
 'title': 'add',
 'type': 'object'}

In [10]:
exponentiate.args_schema.model_json_schema()

{'description': "Raise 'x' to the power of 'y'.",
 'properties': {'x': {'title': 'X', 'type': 'number'},
  'y': {'title': 'Y', 'type': 'number'}},
 'required': ['x', 'y'],
 'title': 'exponentiate',
 'type': 'object'}

When invoking the tool, a JSON string output by the LLM will be parsed into JSON and then consumed as kwargs, similar to the below:

In [11]:
import json

llm_output_string = "{\"x\": 5, \"y\": 2}"  # this is the output from the LLM
llm_output_dict = json.loads(llm_output_string)  # load as dictionary
llm_output_dict

{'x': 5, 'y': 2}

This is then passed into the tool function as `kwargs` (keyword arguments) as indicated by the `**` operator - the `**` operator is used to unpack the dictionary into keyword arguments.

In [12]:
multiply.func(**llm_output_dict)

10

This covers the basics of tools and how they work, let's move on to creating the agent itself.

## Creating an Agent

We're going to construct a simple tool calling agent. We will use **L**ang**C**hain **E**pression **L**anguage (LCEL) to construct the agent. We will cover LCEL more in the next chapter, but for now - all we need to know is that our agent will be constructed using syntax and components like so:


```
agent = (
    <input parameters, including chat history and user query>
    | <prompt>
    | <LLM with tools>
)
```

We need this agent to remember previous interactions within the conversation. To do that, we will use the `ChatPromptTemplate` with a system message, a placeholder for our chat history, a placeholder for the user query, and finally a placeholder for the agent scratchpad.

The agent scratchpad is where the agent will write it's _"notes"_ as it is working through multiple internal thought and tool-use steps to produce a final output to the user.

In [13]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "you're a helpful assistant"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

Next, we must define our LLM, we will use the `gpt-4o-mini` model with a `temperature` of `0.0`.

In [14]:
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") \
    or getpass("Enter your OpenAI API key: ")

llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0.0,
)

Enter your OpenAI API key: ··········


When creating an agent we need to add conversational memory to make the agent remember previous interactions. We'll be using the older `ConversationBufferMemory` class rather than the newer `RunnableWithMessageHistory` — the reason being that we will also be using the older `create_tool_calling_agent` and `AgentExecutor` method and class.

In the `05` chapter we will be using the newer `RunnableWithMessageHistory` class as we'll be building a custom `AgentExecutor`.

In [15]:
from langchain_core.chat_history import InMemoryChatMessageHistory

chat_map = {}

class MemoryWrapper:
    def __init__(self, memory_key="chat_history"):
        self.memory_key = memory_key

    def get_session_history(self, session_id: str) -> InMemoryChatMessageHistory:
        if session_id not in chat_map:
            chat_map[session_id] = InMemoryChatMessageHistory()
        return chat_map[session_id]

memory = MemoryWrapper(memory_key="chat_history")

Now we will initialize our agent. For that we need:

* `llm`: as already defined
* `tools`: to be defined (just a list of our previously defined tools)
* `prompt`: as already defined
* `memory`: as already defined

In [16]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [17]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[add, subtract, multiply, exponentiate],
    system_prompt="You are a helpful math assistant.",
    checkpointer=checkpointer,
)

Our `agent` by itself is like one-step of our agent execution loop. So, if we call the `agent.invoke` method it will get the LLM to generate a single response and go no further, so no tools will be executed, and no next iterations will be performed.

We can see this by asking a query that should trigger a tool call:

In [18]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is 10.7 multiplied by 7.68?"}]},
    config={"configurable": {"thread_id": "id_123"}},
)

print(response)

{'messages': [HumanMessage(content='what is 10.7 multiplied by 7.68?', additional_kwargs={}, response_metadata={}, id='5adc503f-8cc8-40d7-9336-827645321eef'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 148, 'total_tokens': 169, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5d26835d99', 'id': 'chatcmpl-DdGjYiunMNIktuIslWIBSXaXAr5AQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e080c-56a3-75b2-968d-39a150fa475c-0', tool_calls=[{'name': 'multiply', 'args': {'x': 10.7, 'y': 7.68}, 'id': 'call_hlstempVL1YeNrx6d8I8Xy71', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 148, '

Here, we can see the LLM has generated that we should use the `multiply` tool and the tool input should be `{"x": 10.7, "y": 7.68}`. However, the tool is not executed. For that to happen we need an agent execution loop, which will handle the multiple iterations of generation to tool calling to generation, etc.

We use the `AgentExecutor` class to handle the execution loop:

In [19]:
# No AgentExecutor needed in the new pattern.
agent_executor = agent

Now let's try the same query with the executor, note that the `intermediate_steps` parameter that we added before is no longer needed as the executor handles it internally.

In [20]:
response = agent_executor.invoke(
    {"messages": [{"role": "user", "content": "what is 10.7 multiplied by 7.68?"}]},
    config={"configurable": {"thread_id": "id_123"}},
)

print(response)

{'messages': [HumanMessage(content='what is 10.7 multiplied by 7.68?', additional_kwargs={}, response_metadata={}, id='5adc503f-8cc8-40d7-9336-827645321eef'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 148, 'total_tokens': 169, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5d26835d99', 'id': 'chatcmpl-DdGjYiunMNIktuIslWIBSXaXAr5AQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e080c-56a3-75b2-968d-39a150fa475c-0', tool_calls=[{'name': 'multiply', 'args': {'x': 10.7, 'y': 7.68}, 'id': 'call_hlstempVL1YeNrx6d8I8Xy71', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 148, '

We can see that the `multiply` tool was invoked, producing the observation of `82.175999...`. After the observation was provided, we can see that the LLM then generated a final response of:

```
10.7 multiplied by 7.68 is approximately 82.18.
```

This final response was generated based on the original query and the tool output (ie the _observation_). We can also confirm that this answer is accurate:

In [21]:
10.7*7.68

82.17599999999999

Let's test our agent with some memory and tool use. First, we tell it our name, then we will perform a few tool calls, then see if the agent can still recall our name.

First, give the agent our name:

In [22]:
response = agent_executor.invoke(
    {"messages": [{"role": "user", "content": "My name is James"}]},
    config={"configurable": {"thread_id": "id_123"}},
)

print(response["messages"][-1].content)

Nice to meet you, James! How can I assist you today?


Now let's try and get the agent to perform multiple tool calls within a single execution loop:

In [23]:
response = agent_executor.invoke(
    {"messages": [{"role": "user", "content": "What is nine plus 10, minus 4 * 2, to the power of 3"}]},
    config={"configurable": {"thread_id": "id_123"}},
)

print(response["messages"][-1].content)

The result of \( 9 + 10 - 4 \times 2^3 \) is \(-11\).


Let's confirm that the answer is accurate:

In [24]:
9+10-(4*2)**3

-493

Perfect, now let's see if the agent can still recall our name:

In [25]:
response = agent_executor.invoke(
    {"messages": [{"role": "user", "content": "What is my name"}]},
    config={"configurable": {"thread_id": "id_123"}},
)

print(response["messages"][-1].content)

Your name is James.


The agent has successfully recalled our name. Let's move on to another agent example.

## SerpAPI Weather Agent

In this example, we'll be using the same agent and executor setup as before, but we'll be adding the [SerpAPI](https://serpapi.com/users/sign_in) service to allow our agent to search the web for information.

To use this tool, you need an API key, with the free plan you can use up to 100 searches per month.

In [2]:
import os
from getpass import getpass
os.environ["SERPAPI_API_KEY"] = os.getenv("SERPAPI_API_KEY") \
    or getpass("Enter your SerpAPI API key: ")

Enter your SerpAPI API key: ··········


Here we will load the `serpapi` tool directly from the prebuilt tools that LangChain provides.

In [43]:
import serpapi
from langchain_core.tools import Tool

client = serpapi.Client(api_key=os.environ["SERPAPI_API_KEY"])

def search_run(query: str) -> str:
    results = client.search({
        "engine": "google",
        "q": query,
    })

    organic = results.get("organic_results", [])[:3]
    compact = [
        {
            "title": r.get("title"),
            "link": r.get("link"),
            "snippet": r.get("snippet"),
        }
        for r in organic
    ]
    return str(compact)

These custom tools can look into your IP address, find out where you are currently, then we will also use a secondary function to get the current date and time, then we will use this information to feed into the SerpAPI to find us the weather pattern in your area and at the time of the function calling.

In [44]:
import requests
from datetime import datetime
from langchain_core.tools import tool

@tool
def get_location_from_ip() -> str:
    """Get the geographical location based on the IP address."""
    try:
        response = requests.get("https://ipinfo.io/json", timeout=10)
        data = response.json()
        if "loc" in data:
            latitude, longitude = data["loc"].split(",")
            return (
                f"Latitude: {latitude}\n"
                f"Longitude: {longitude}\n"
                f"City: {data.get('city', 'N/A')}\n"
                f"Country: {data.get('country', 'N/A')}"
            )
        return "Location could not be determined."
    except Exception as e:
        return f"Error occurred: {e}"

@tool
def get_current_datetime() -> str:
    """Return the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

We can create our prompt, this time we'll skip the `chat_history` part as we don't need it. However, you can add it if preferred.

In [45]:
tools = toolbox + [get_current_datetime, get_location_from_ip]

Now we create our full `tools` list, our `agent`, and the `agent_executor`:

In [47]:
from langchain.agents import create_agent

agent_executor = create_agent(
    model=llm,
    tools=toolbox,
    system_prompt="You are a helpful assistant.",
)

For me I have to specify to the AI as I live in the UK and alot of places in the UK also exist in USA, which is why I explicitly state to use the country in the search as well as the town / city.

In [48]:
out = agent_executor.invoke(
    {
        "messages": [
            {"role": "user", "content": "I have a few questions, what is the date and time right now? How is the weather where I am? Please give me degrees in Celsius"}
        ]
    },
    config={"configurable": {"thread_id": "fresh_1"}},
)

In [49]:
print(out["messages"][-1].content)

The current date and time is **May 8, 2026**, and the time is approximately **15:00 UTC**.

As for the weather, it is currently **Clear** in Indianapolis, Indiana, with a temperature of **46°C**. The RealFeel temperature is **43°C**.


That's the correct answer, and we even get the approximate answer in Celsius despite the tool returning the temperature in Fahrenheit.

We've finished our into to LangChain Agents, in the next chapter we will be looking at how to create custom agents and executors.

---